# Rotation validation: phase correlation on opposing projections

Estimate the centre of rotation from two I12 dataset 3 projections acquired
180 degrees apart. Apply dark/flat correction and the negative logarithm,
select detector rows `200:1750`, and compare the centre against 1253.5 pixels.

## Data and execution

Use the named dataset archives from [Zenodo record 17252190](https://zenodo.org/records/17252190).
The repository downloader verifies the archive checksums.
Run from the repository root:

```bash
python .scripts/download_zenodo.py Zenodo
python -m pip install -e '.[validation]'
python -m pytest --nbmake --nbmake-timeout=1800 --overwrite docs/source/validation/rotation/center_pc.ipynb
```

Use an NVIDIA GPU with compatible CuPy and ToMoBAR installed, plus ASTRA for
ASTRA-based reconstruction. Run notebooks serially. Set `HTTOMOLIBGPU_DATA_DIR`
to use an archive stored elsewhere.

Documentation builds render saved outputs without running GPU computations.
Execute with `--overwrite` before building documentation to update the results.


In [ ]:
import gc
import hashlib
import json
import os
import platform
from importlib.metadata import version
from pathlib import Path

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np

from httomolibgpu.prep.normalize import dark_flat_field_correction, minus_log
from httomolibgpu.recon.rotation import find_center_vo, find_center_pc, find_center_360

repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "httomolibgpu").is_dir()
    ),
    None,
)
configured_dir = os.environ.get("HTTOMOLIBGPU_DATA_DIR")
if configured_dir:
    data_dir = Path(configured_dir).expanduser()
elif repo_root is not None:
    data_dir = repo_root / "Zenodo"
else:
    raise FileNotFoundError("Set HTTOMOLIBGPU_DATA_DIR to the Zenodo archive directory")

props = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
environment = {
    "python": platform.python_version(),
    "packages": {
        name: version(name) for name in ("httomolibgpu", "numpy", "matplotlib")
    },
    "cupy": cp.__version__,
    "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
    "cuda_driver": cp.cuda.runtime.driverGetVersion(),
    "gpu": (
        props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    ),
}
print(json.dumps(environment, indent=2))
results = []
dataset_provenance = {}


def clean_memory():
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()


def archive_path(name):
    path = data_dir / (name + ".npz")
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}; run .scripts/download_zenodo.py first"
        )
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    dataset_provenance[name] = {
        "archive": path.name,
        "sha256": digest.hexdigest(),
        "zenodo_record": "17252190",
    }
    print(json.dumps(dataset_provenance[name], indent=2))
    return path


def load_transmission(name):
    with np.load(archive_path(name)) as archive:
        raw = cp.asarray(archive["projdata"])
        flats = cp.asarray(archive["flats"])
        darks = cp.asarray(archive["darks"])
        print("Raw, flats, darks:", raw.shape, flats.shape, darks.shape)
    transmission = dark_flat_field_correction(raw, flats, darks)
    del raw, flats, darks
    clean_memory()
    return transmission


def display_sinogram(sinogram, observed, expected, title):
    image = cp.asnumpy(sinogram)
    lo, hi = np.percentile(image[np.isfinite(image)], [1, 99])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    im = axes[0].imshow(image, cmap="gray", aspect="auto", vmin=lo, vmax=hi)
    axes[0].axvline(float(observed), color="tab:orange", label="Observed centre")
    axes[0].axvline(
        float(expected), color="tab:cyan", linestyle="--", label="Reference centre"
    )
    axes[0].set(
        title=title, xlabel="Detector column (pixel)", ylabel="Projection index"
    )
    axes[0].legend(fontsize=8)
    fig.colorbar(im, ax=axes[0], label="Corrected transmission")
    axes[1].plot(np.mean(image, axis=0))
    axes[1].axvline(float(observed), color="tab:orange")
    axes[1].axvline(float(expected), color="tab:cyan", linestyle="--")
    axes[1].set(
        title="Angular mean profile",
        xlabel="Detector column (pixel)",
        ylabel="Mean transmission",
    )
    plt.show()
    plt.close(fig)


def check_results():
    print(json.dumps(results, indent=2))
    failed = [row for row in results if not row["passed"]]
    assert not failed, "Rotation validation checks failed: " + json.dumps(
        failed, indent=2
    )
    print(f"All {len(results)} rotation validation checks passed.")

## Preprocessing and centre estimate

Load `proj1`, `proj2`, `flats` and `darks` from the archive. Apply dark/flat
correction and the negative logarithm, then select rows `200:1750` before
estimating the centre. Compare against 1253.5 pixels and float32 dtype.


In [ ]:
with np.load(archive_path("i12_dataset3")) as archive:
    proj1 = cp.asarray(archive["proj1"])
    proj2 = cp.asarray(archive["proj2"])
    flats = cp.asarray(archive["flats"])
    darks = cp.asarray(archive["darks"])
raw = cp.empty((2, *proj1.shape), dtype=cp.float32)
raw[0], raw[1] = proj1, proj2
print("Raw and calibration shapes:", raw.shape, flats.shape, darks.shape)
normalised = minus_log(dark_flat_field_correction(raw, flats, darks))
first = normalised[0, 200:1750, :]
second = normalised[1, 200:1750, :]
del proj1, proj2, raw, flats, darks, normalised
clean_memory()
cor = find_center_pc(first, second)
passed = bool(cor == 1253.5 and cor.dtype == np.float32)
results.append(
    {
        "test": "test_center_pc_i12_dataset3",
        "dataset": "i12_dataset3",
        "crop": "200:1750",
        "observed": float(cor),
        "expected": 1253.5,
        "dtype": str(cor.dtype),
        "expected_dtype": "float32",
        "passed": passed,
    }
)
print(json.dumps(results[-1], indent=2))

## Opposing images and horizontal registration

Mirror the second projection in detector columns. For a width `W`, the measured
centre gives the horizontal translation `2 * cor - (W - 1)`. The visual comparison
below applies that horizontal translation only; vertical motion and differences
between the opposing views can remain. This plot is additional interpretation,
not an extra modification of the centre-finding call.


In [ ]:
from cupyx.scipy.ndimage import shift

mirrored = cp.fliplr(second)
horizontal_shift = 2 * float(cor) - (first.shape[1] - 1)
aligned = shift(mirrored, (0, horizontal_shift), order=3, mode="constant", cval=0)
a, b, c = (cp.asnumpy(value) for value in (first, mirrored, aligned))
lo, hi = np.percentile(a[np.isfinite(a)], [1, 99])
residual = np.abs(a - c)
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for ax, image, title in zip(
    axes.ravel()[:3],
    (a, b, c),
    (
        "First projection, rows 200:1750",
        "Mirrored opposing projection",
        "After horizontal registration",
    ),
):
    ax.imshow(image, cmap="gray", vmin=lo, vmax=hi, aspect="auto")
    ax.set(title=title, xlabel="Detector column", ylabel="Selected detector row")
im = axes[1, 1].imshow(
    residual,
    cmap="magma",
    aspect="auto",
    vmin=0,
    vmax=np.percentile(residual[np.isfinite(residual)], 99),
)
axes[1, 1].set(
    title="Absolute residual after horizontal registration",
    xlabel="Detector column",
    ylabel="Selected detector row",
)
fig.colorbar(im, ax=axes[1, 1], label="Absolute line-integral difference")
plt.show()
plt.close(fig)
print("Derived horizontal shift (pixels):", horizontal_shift)

## Final validation


In [ ]:
check_results()
del first, second, mirrored, aligned, a, b, c, residual
clean_memory()